In [1]:
"""
PHASE 2+3 - CELL 1: LOAD HOURLY DATA AND SETUP
Load the downloaded 2021-2025 hourly weather data
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("="*70)
print("PHASE 2+3: FEATURE ENGINEERING & TARGET VARIABLE DESIGN")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Load hourly data
print("Loading hourly weather data (2021-2025)...")
df = pd.read_csv('data/colombo_hourly_raw_2021_2025.csv', parse_dates=['datetime'])

# Sort by datetime (CRITICAL for time series!)
df = df.sort_values('datetime').reset_index(drop=True)

print(f"✓ Data loaded and sorted")
print(f"  Records: {len(df):,}")
print(f"  Date range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"  Variables: {list(df.columns)}")

# Extract datetime components
df['date'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
df['year'] = df['datetime'].dt.year
df['day_of_year'] = df['datetime'].dt.dayofyear
df['day_of_week'] = df['datetime'].dt.dayofweek

print(f"\n✓ Datetime components extracted")
print(f"\nFirst 3 records:")
print(df[['datetime', 'hour', 'temperature_2m', 'precipitation']].head(3))


PHASE 2+3: FEATURE ENGINEERING & TARGET VARIABLE DESIGN
Start time: 2026-02-18 13:33:45

Loading hourly weather data (2021-2025)...
✓ Data loaded and sorted
  Records: 43,824
  Date range: 2021-01-01 00:00:00 to 2025-12-31 23:00:00
  Variables: ['datetime', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'precipitation', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'year']

✓ Datetime components extracted

First 3 records:
             datetime  hour  temperature_2m  precipitation
0 2021-01-01 00:00:00     0            24.1            0.0
1 2021-01-01 01:00:00     1            23.7            0.0
2 2021-01-01 02:00:00     2            23.2            0.0


In [2]:
"""
CELL 2: CALCULATE 24-HOUR ROLLING TARGET VARIABLE
For each hour H, calculate total rainfall from H+1 to H+24 (next 24 hours)
"""

print("\n" + "="*70)
print("CALCULATING 24-HOUR ROLLING RAINFALL TARGET")
print("="*70)

print("\nCalculating target for each hour...")
print("Target = Sum of precipitation in next 24 hours (H+1 to H+24)")

# Calculate target using rolling sum
target_24h = []

for i in range(len(df)):
    if i + 24 < len(df):
        # Sum next 24 hours of rainfall (rows i+1 to i+24)
        next_24h_rain = df.iloc[i+1:i+25]['precipitation'].sum()
        target_24h.append(next_24h_rain)
    else:
        # Last 24 hours don't have complete future window
        target_24h.append(np.nan)

df['target_24h_rainfall'] = target_24h

# Check for valid targets
valid_targets = df['target_24h_rainfall'].notna()
invalid_count = (~valid_targets).sum()

print(f"\n✓ Target variable calculated")
print(f"  Valid samples: {valid_targets.sum():,}")
print(f"  Invalid samples (no future data): {invalid_count}")

# Statistics
print(f"\nTarget variable statistics:")
print(f"  Min: {df['target_24h_rainfall'].min():.2f} mm")
print(f"  Max: {df['target_24h_rainfall'].max():.2f} mm")
print(f"  Mean: {df['target_24h_rainfall'].mean():.2f} mm")
print(f"  Median: {df['target_24h_rainfall'].median():.2f} mm")
print(f"  Std: {df['target_24h_rainfall'].std():.2f} mm")

# Zero-inflation
zero_count = (df['target_24h_rainfall'] == 0).sum()
zero_pct = zero_count / valid_targets.sum() * 100
print(f"\n  Zero rainfall windows: {zero_count:,} ({zero_pct:.1f}%)")
print(f"  Rainy windows: {valid_targets.sum() - zero_count:,} ({100-zero_pct:.1f}%)")

# Sample verification (IMPORTANT: Check no data leakage!)
print(f"\n SAMPLE VERIFICATION (Check for data leakage):")
sample = df[['datetime', 'precipitation', 'target_24h_rainfall']].head(3)
print(sample.to_string(index=False))
print("\n✓ Verify: target_24h_rainfall should NOT include current hour's precipitation")



CALCULATING 24-HOUR ROLLING RAINFALL TARGET

Calculating target for each hour...
Target = Sum of precipitation in next 24 hours (H+1 to H+24)

✓ Target variable calculated
  Valid samples: 43,800
  Invalid samples (no future data): 24

Target variable statistics:
  Min: 0.00 mm
  Max: 239.20 mm
  Mean: 8.49 mm
  Median: 4.30 mm
  Std: 11.82 mm

  Zero rainfall windows: 3,156 (7.2%)
  Rainy windows: 40,644 (92.8%)

 SAMPLE VERIFICATION (Check for data leakage):
           datetime  precipitation  target_24h_rainfall
2021-01-01 00:00:00            0.0                  4.7
2021-01-01 01:00:00            0.0                  4.7
2021-01-01 02:00:00            0.0                  4.7

✓ Verify: target_24h_rainfall should NOT include current hour's precipitation


In [3]:
"""
CELL 3: CREATE ROLLING 24-HOUR WINDOW FEATURES
Calculate statistics over the previous 24 hours for all weather variables
"""

print("\n" + "="*70)
print("CREATING ROLLING 24-HOUR WINDOW FEATURES")
print("="*70)
print("\nNote: These use PAST 24 hours (lookback), not future!")

# Function to create rolling features
def create_rolling_features(df, window=24):
    """
    Create rolling window statistics for weather variables
    
    Parameters:
    -----------
    df : DataFrame
        Hourly weather data
    window : int
        Rolling window size in hours (default: 24)
    
    Returns:
    --------
    DataFrame with new rolling features
    """
    
    print(f"\nCreating rolling {window}h features...")
    
    # PRECIPITATION (Last 24h)
    print(f"\n  1. Precipitation features...")
    df[f'precip_{window}h_sum'] = df['precipitation'].rolling(window=window, min_periods=int(window*0.75)).sum()
    df[f'precip_{window}h_max'] = df['precipitation'].rolling(window=window, min_periods=int(window*0.75)).max()
    df[f'precip_{window}h_mean'] = df['precipitation'].rolling(window=window, min_periods=int(window*0.75)).mean()
    df[f'precip_{window}h_std'] = df['precipitation'].rolling(window=window, min_periods=int(window*0.75)).std()
    
    # Count rainy hours (precipitation > 0.1 mm)
    df[f'precip_{window}h_rainy_hours'] = df['precipitation'].rolling(window=window, min_periods=int(window*0.75)).apply(
        lambda x: (x > 0.1).sum(), raw=True
    )
    
    # TEMPERATURE (Last 24h)
    print(f"  2. Temperature features...")
    df[f'temp_{window}h_mean'] = df['temperature_2m'].rolling(window=window, min_periods=int(window*0.75)).mean()
    df[f'temp_{window}h_max'] = df['temperature_2m'].rolling(window=window, min_periods=int(window*0.75)).max()
    df[f'temp_{window}h_min'] = df['temperature_2m'].rolling(window=window, min_periods=int(window*0.75)).min()
    df[f'temp_{window}h_range'] = df[f'temp_{window}h_max'] - df[f'temp_{window}h_min']
    df[f'temp_{window}h_std'] = df['temperature_2m'].rolling(window=window, min_periods=int(window*0.75)).std()
    
    # Temperature trend (change over 24h)
    df[f'temp_{window}h_trend'] = df['temperature_2m'] - df['temperature_2m'].shift(window)
    
    # HUMIDITY (Last 24h)
    print(f"  3. Humidity features...")
    df[f'humidity_{window}h_mean'] = df['relative_humidity_2m'].rolling(window=window, min_periods=int(window*0.75)).mean()
    df[f'humidity_{window}h_max'] = df['relative_humidity_2m'].rolling(window=window, min_periods=int(window*0.75)).max()
    df[f'humidity_{window}h_min'] = df['relative_humidity_2m'].rolling(window=window, min_periods=int(window*0.75)).min()
    df[f'humidity_{window}h_range'] = df[f'humidity_{window}h_max'] - df[f'humidity_{window}h_min']
    
    # Count hours with high humidity (>80%)
    df[f'humidity_{window}h_hours_above_80'] = df['relative_humidity_2m'].rolling(window=window, min_periods=int(window*0.75)).apply(
        lambda x: (x > 80).sum(), raw=True
    )
    
    # PRESSURE (Last 24h)
    print(f"  4. Pressure features...")
    df[f'pressure_{window}h_mean'] = df['surface_pressure'].rolling(window=window, min_periods=int(window*0.75)).mean()
    df[f'pressure_{window}h_min'] = df['surface_pressure'].rolling(window=window, min_periods=int(window*0.75)).min()
    df[f'pressure_{window}h_max'] = df['surface_pressure'].rolling(window=window, min_periods=int(window*0.75)).max()
    df[f'pressure_{window}h_std'] = df['surface_pressure'].rolling(window=window, min_periods=int(window*0.75)).std()
    
    # Pressure trend (change over 24h)
    df[f'pressure_{window}h_trend'] = df['surface_pressure'] - df['surface_pressure'].shift(window)
    
    # WIND (Last 24h)
    print(f"  5. Wind features...")
    df[f'wind_{window}h_mean'] = df['wind_speed_10m'].rolling(window=window, min_periods=int(window*0.75)).mean()
    df[f'wind_{window}h_max'] = df['wind_speed_10m'].rolling(window=window, min_periods=int(window*0.75)).max()
    df[f'wind_{window}h_std'] = df['wind_speed_10m'].rolling(window=window, min_periods=int(window*0.75)).std()
    
    # CLOUD COVER (Last 24h)
    print(f"  6. Cloud cover features...")
    df[f'cloud_{window}h_mean'] = df['cloud_cover'].rolling(window=window, min_periods=int(window*0.75)).mean()
    df[f'cloud_{window}h_max'] = df['cloud_cover'].rolling(window=window, min_periods=int(window*0.75)).max()
    
    # Count overcast hours (cloud cover > 90%)
    df[f'cloud_{window}h_hours_above_90'] = df['cloud_cover'].rolling(window=window, min_periods=int(window*0.75)).apply(
        lambda x: (x > 90).sum(), raw=True
    )
    
    # DEW POINT (Last 24h)
    print(f"  7. Dew point features...")
    df[f'dewpoint_{window}h_mean'] = df['dew_point_2m'].rolling(window=window, min_periods=int(window*0.75)).mean()
    
    return df

# Apply rolling features
df = create_rolling_features(df, window=24)

# Count new features
rolling_features = [col for col in df.columns if '24h' in col and col != 'target_24h_rainfall']
print(f"\n✓ Rolling 24h features created: {len(rolling_features)} features")
print(f"\nExample features: {rolling_features[:5]}")



CREATING ROLLING 24-HOUR WINDOW FEATURES

Note: These use PAST 24 hours (lookback), not future!

Creating rolling 24h features...

  1. Precipitation features...
  2. Temperature features...
  3. Humidity features...
  4. Pressure features...
  5. Wind features...
  6. Cloud cover features...
  7. Dew point features...

✓ Rolling 24h features created: 28 features

Example features: ['precip_24h_sum', 'precip_24h_max', 'precip_24h_mean', 'precip_24h_std', 'precip_24h_rainy_hours']


In [4]:
"""
CELL 4: CREATE CURRENT HOUR STATE FEATURES
Features representing the exact conditions at prediction time
"""

print("\n" + "="*70)
print("CREATING CURRENT HOUR STATE FEATURES")
print("="*70)

# Current state (these are already in df, just rename for clarity)
print("\n1. Current meteorological conditions...")
df['temp_current'] = df['temperature_2m']
df['humidity_current'] = df['relative_humidity_2m']
df['pressure_current'] = df['surface_pressure']
df['dewpoint_current'] = df['dew_point_2m']
df['wind_speed_current'] = df['wind_speed_10m']
df['wind_direction_current'] = df['wind_direction_10m']
df['cloud_cover_current'] = df['cloud_cover']

# Dew point depression (CRITICAL FEATURE!)
print("\n2. Dew point depression (saturation indicator)...")
df['dew_point_depression_current'] = df['temp_current'] - df['dewpoint_current']
print("   ✓ dew_point_depression_current (low value = near saturation = rain likely)")

# Cyclical encoding for hour of day
print("\n3. Cyclical encoding of hour of day...")
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
print("   ✓ hour_sin, hour_cos (captures diurnal cycle)")

current_features = ['temp_current', 'humidity_current', 'pressure_current', 
                   'dewpoint_current', 'dew_point_depression_current',
                   'wind_speed_current', 'wind_direction_current', 
                   'cloud_cover_current', 'hour_sin', 'hour_cos']

print(f"\n✓ Current hour state features created: {len(current_features)} features")



CREATING CURRENT HOUR STATE FEATURES

1. Current meteorological conditions...

2. Dew point depression (saturation indicator)...
   ✓ dew_point_depression_current (low value = near saturation = rain likely)

3. Cyclical encoding of hour of day...
   ✓ hour_sin, hour_cos (captures diurnal cycle)

✓ Current hour state features created: 10 features


In [5]:
"""
CELL 5: CREATE DAILY AGGREGATES
Aggregate hourly data to daily (midnight-midnight) for creating daily lag features
"""

print("\n" + "="*70)
print("CREATING DAILY AGGREGATES")
print("="*70)

print("\nAggregating hourly data to daily summaries...")

# Convert date to datetime for grouping
df['date_dt'] = pd.to_datetime(df['date'])

# Group by date and aggregate
daily_agg = df.groupby('date_dt').agg({
    # Precipitation
    'precipitation': ['sum', 'max', lambda x: (x > 0.1).sum()],
    # Temperature
    'temperature_2m': ['mean', 'max', 'min'],
    # Humidity
    'relative_humidity_2m': ['mean', 'max', 'min'],
    # Pressure
    'surface_pressure': ['mean', 'min', 'max'],
    # Wind
    'wind_speed_10m': ['mean', 'max'],
    # Cloud
    'cloud_cover': ['mean', 'max'],
    # Dew point
    'dew_point_2m': ['mean']
}).reset_index()

# Flatten multi-level columns
daily_agg.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                      for col in daily_agg.columns.values]

# Rename for clarity
daily_agg.columns = [
    'date_dt',
    'daily_rainfall_total', 'daily_rainfall_max_hourly', 'daily_rainy_hours',
    'daily_temp_mean', 'daily_temp_max', 'daily_temp_min',
    'daily_humidity_mean', 'daily_humidity_max', 'daily_humidity_min',
    'daily_pressure_mean', 'daily_pressure_min', 'daily_pressure_max',
    'daily_wind_mean', 'daily_wind_max',
    'daily_cloud_mean', 'daily_cloud_max',
    'daily_dewpoint_mean'
]

# Derived daily features
daily_agg['daily_temp_range'] = daily_agg['daily_temp_max'] - daily_agg['daily_temp_min']

print(f"✓ Daily aggregates created")
print(f"  Daily records: {len(daily_agg):,}")
print(f"  Daily features: {len(daily_agg.columns) - 1}")

print(f"\nSample (first 3 days):")
print(daily_agg.head(3).to_string(index=False))



CREATING DAILY AGGREGATES

Aggregating hourly data to daily summaries...
✓ Daily aggregates created
  Daily records: 1,826
  Daily features: 18

Sample (first 3 days):
   date_dt  daily_rainfall_total  daily_rainfall_max_hourly  daily_rainy_hours  daily_temp_mean  daily_temp_max  daily_temp_min  daily_humidity_mean  daily_humidity_max  daily_humidity_min  daily_pressure_mean  daily_pressure_min  daily_pressure_max  daily_wind_mean  daily_wind_max  daily_cloud_mean  daily_cloud_max  daily_dewpoint_mean  daily_temp_range
2021-01-01                   4.7                        3.6                  4        24.975000            28.6            22.5            89.125000                  98                  74          1008.970833              1007.6              1011.0         7.262500            14.0         86.375000              100            22.970833               6.1
2021-01-02                   1.5                        0.4                  5        25.929167            29.6      

In [6]:
"""
CELL 6: CREATE DAILY LAG FEATURES
Generate lag features from daily aggregates (yesterday, last week, etc.)
"""

print("\n" + "="*70)
print("CREATING DAILY LAG FEATURES")
print("="*70)

# Sort by date
daily_agg = daily_agg.sort_values('date_dt').reset_index(drop=True)

# 1. RAINFALL LAGS
print("\n1. Rainfall lag features...")
lags = [1, 2, 3, 7, 14]
for lag in lags:
    daily_agg[f'rainfall_lag_{lag}d'] = daily_agg['daily_rainfall_total'].shift(lag)
    print(f"   ✓ rainfall_lag_{lag}d")

# 2. CUMULATIVE RAINFALL
print("\n2. Cumulative rainfall features...")
windows = [3, 7, 14, 30]
for window in windows:
    daily_agg[f'rainfall_cumsum_{window}d'] = daily_agg['daily_rainfall_total'].rolling(
        window=window, min_periods=1
    ).sum()
    print(f"   ✓ rainfall_cumsum_{window}d")

# 3. ROLLING MEAN RAINFALL
print("\n3. Rolling mean rainfall features...")
for window in [7, 14, 30]:
    daily_agg[f'rainfall_rolling_mean_{window}d'] = daily_agg['daily_rainfall_total'].rolling(
        window=window, min_periods=1
    ).mean()
    print(f"   ✓ rainfall_rolling_mean_{window}d")

# 4. CONSECUTIVE DRY/RAINY DAYS
print("\n4. Consecutive day features...")

# Consecutive dry days
consecutive_dry = []
count = 0
for rain in daily_agg['daily_rainfall_total']:
    if rain < 0.1:  # Essentially dry
        count += 1
    else:
        count = 0
    consecutive_dry.append(count)
daily_agg['consecutive_dry_days'] = consecutive_dry
print(f"   ✓ consecutive_dry_days")

# Consecutive rainy days
consecutive_rainy = []
count = 0
for rain in daily_agg['daily_rainfall_total']:
    if rain >= 0.1:  # Has rain
        count += 1
    else:
        count = 0
    consecutive_rainy.append(count)
daily_agg['consecutive_rainy_days'] = consecutive_rainy
print(f"   ✓ consecutive_rainy_days")

# Days since last rain
days_since = []
count = 0
for rain in daily_agg['daily_rainfall_total']:
    if rain >= 0.1:
        count = 0
    else:
        count += 1
    days_since.append(count)
daily_agg['days_since_last_rain'] = days_since
print(f"   ✓ days_since_last_rain")

# 5. TEMPERATURE LAGS AND TRENDS
print("\n5. Temperature lag and trend features...")
for lag in [1, 3, 7]:
    daily_agg[f'temp_lag_{lag}d'] = daily_agg['daily_temp_mean'].shift(lag)
    print(f"   ✓ temp_lag_{lag}d")

daily_agg['temp_trend_7d'] = daily_agg['daily_temp_mean'] - daily_agg['daily_temp_mean'].shift(7)
daily_agg['temp_trend_14d'] = daily_agg['daily_temp_mean'] - daily_agg['daily_temp_mean'].shift(14)
print(f"   ✓ temp_trend_7d, temp_trend_14d")

# 6. PRESSURE LAGS AND CHANGES
print("\n6. Pressure lag and change features...")
for lag in [1, 3]:
    daily_agg[f'pressure_lag_{lag}d'] = daily_agg['daily_pressure_mean'].shift(lag)
    print(f"   ✓ pressure_lag_{lag}d")

daily_agg['pressure_change_3d'] = daily_agg['daily_pressure_mean'] - daily_agg['daily_pressure_mean'].shift(3)
daily_agg['pressure_change_7d'] = daily_agg['daily_pressure_mean'] - daily_agg['daily_pressure_mean'].shift(7)
print(f"   ✓ pressure_change_3d, pressure_change_7d")

# 7. HUMIDITY LAGS
print("\n7. Humidity lag features...")
for lag in [1, 3]:
    daily_agg[f'humidity_lag_{lag}d'] = daily_agg['daily_humidity_mean'].shift(lag)
    print(f"   ✓ humidity_lag_{lag}d")

daily_lag_features = [col for col in daily_agg.columns if 'lag' in col or 'cumsum' in col 
                      or 'rolling_mean' in col or 'consecutive' in col or 'days_since' in col
                      or 'trend' in col or 'change' in col]

print(f"\n✓ Daily lag features created: {len(daily_lag_features)} features")



CREATING DAILY LAG FEATURES

1. Rainfall lag features...
   ✓ rainfall_lag_1d
   ✓ rainfall_lag_2d
   ✓ rainfall_lag_3d
   ✓ rainfall_lag_7d
   ✓ rainfall_lag_14d

2. Cumulative rainfall features...
   ✓ rainfall_cumsum_3d
   ✓ rainfall_cumsum_7d
   ✓ rainfall_cumsum_14d
   ✓ rainfall_cumsum_30d

3. Rolling mean rainfall features...
   ✓ rainfall_rolling_mean_7d
   ✓ rainfall_rolling_mean_14d
   ✓ rainfall_rolling_mean_30d

4. Consecutive day features...
   ✓ consecutive_dry_days
   ✓ consecutive_rainy_days
   ✓ days_since_last_rain

5. Temperature lag and trend features...
   ✓ temp_lag_1d
   ✓ temp_lag_3d
   ✓ temp_lag_7d
   ✓ temp_trend_7d, temp_trend_14d

6. Pressure lag and change features...
   ✓ pressure_lag_1d
   ✓ pressure_lag_3d
   ✓ pressure_change_3d, pressure_change_7d

7. Humidity lag features...
   ✓ humidity_lag_1d
   ✓ humidity_lag_3d

✓ Daily lag features created: 26 features


In [7]:
"""
CELL 7: MERGE DAILY LAGS BACK TO HOURLY DATA
Join daily lag features to each hourly record
"""

print("\n" + "="*70)
print("MERGING DAILY LAGS TO HOURLY DATA")
print("="*70)

# Ensure date columns are same type
df['date_dt'] = pd.to_datetime(df['date'])

print(f"\nBefore merge:")
print(f"  Hourly records: {len(df):,}")
print(f"  Hourly columns: {len(df.columns)}")

# Merge daily aggregates to hourly
df = df.merge(daily_agg, on='date_dt', how='left', suffixes=('', '_daily'))

print(f"\nAfter merge:")
print(f"  Merged records: {len(df):,}")
print(f"  Merged columns: {len(df.columns)}")
print(f"  New columns added: {len(df.columns) - len([c for c in df.columns if not c.endswith('_daily')])}")

# Check for missing after merge
missing_after = df.isnull().sum().sum()
print(f"\n  Missing values after merge: {missing_after:,}")

print(f"\n✓ Daily lags successfully merged to hourly data")



MERGING DAILY LAGS TO HOURLY DATA

Before merge:
  Hourly records: 43,824
  Hourly columns: 55

After merge:
  Merged records: 43,824
  Merged columns: 99
  New columns added: 0

  Missing values after merge: 2,362

✓ Daily lags successfully merged to hourly data


In [8]:
"""
CELL 8: CREATE TEMPORAL AND SEASONAL FEATURES
Cyclical time encoding and monsoon season indicators
"""

print("\n" + "="*70)
print("CREATING TEMPORAL AND SEASONAL FEATURES")
print("="*70)

# 1. CYCLICAL ENCODING
print("\n1. Cyclical time encoding...")

# Month (1-12)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
print("   ✓ month_sin, month_cos")

# Day of year (1-365)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
print("   ✓ day_of_year_sin, day_of_year_cos")

# 2. MONSOON SEASON INDICATORS
print("\n2. Monsoon season indicators (Sri Lanka specific)...")

def get_monsoon_season(month):
    """
    Classify month into monsoon seasons for Sri Lanka
    SW Monsoon: May-July (5-7)
    NE Monsoon: December-February (12, 1, 2)
    Inter-Monsoon Heavy: April, October-November (4, 10, 11)
    Inter-Monsoon Light: March, August-September (3, 8, 9)
    """
    if month in [5, 6, 7]:
        return 'SW_Monsoon'
    elif month in [12, 1, 2]:
        return 'NE_Monsoon'
    elif month in [4, 10, 11]:
        return 'Inter_Heavy'
    else:  # 3, 8, 9
        return 'Inter_Light'

df['monsoon_season'] = df['month'].apply(get_monsoon_season)

# One-hot encode monsoon seasons
monsoon_dummies = pd.get_dummies(df['monsoon_season'], prefix='season')
df = pd.concat([df, monsoon_dummies], axis=1)

print(f"   ✓ Monsoon seasons created:")
for season in monsoon_dummies.columns:
    count = df[season].sum()
    print(f"     - {season}: {count:,} hours")

# 3. DAYS INTO MONSOON
print("\n3. Days into monsoon season...")

def days_into_sw_monsoon(row):
    """Days since May 1 if in SW monsoon, else 0"""
    if row['month'] in [5, 6, 7]:
        may_1 = pd.Timestamp(year=row['year'], month=5, day=1)
        return (row['datetime'] - may_1).days
    return 0

def days_into_ne_monsoon(row):
    """Days since Dec 1 if in NE monsoon, else 0"""
    if row['month'] in [12, 1, 2]:
        if row['month'] == 12:
            dec_1 = pd.Timestamp(year=row['year'], month=12, day=1)
        else:  # Jan or Feb
            dec_1 = pd.Timestamp(year=row['year']-1, month=12, day=1)
        return (row['datetime'] - dec_1).days
    return 0

df['days_into_sw_monsoon'] = df.apply(days_into_sw_monsoon, axis=1)
df['days_into_ne_monsoon'] = df.apply(days_into_ne_monsoon, axis=1)
print("   ✓ days_into_sw_monsoon, days_into_ne_monsoon")

temporal_features = ['month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos',
                    'season_Inter_Heavy', 'season_Inter_Light', 'season_NE_Monsoon', 
                    'season_SW_Monsoon', 'days_into_sw_monsoon', 'days_into_ne_monsoon']

print(f"\n✓ Temporal/seasonal features created: {len(temporal_features)} features")



CREATING TEMPORAL AND SEASONAL FEATURES

1. Cyclical time encoding...
   ✓ month_sin, month_cos
   ✓ day_of_year_sin, day_of_year_cos

2. Monsoon season indicators (Sri Lanka specific)...
   ✓ Monsoon seasons created:
     - season_Inter_Heavy: 10,920 hours
     - season_Inter_Light: 11,040 hours
     - season_NE_Monsoon: 10,824 hours
     - season_SW_Monsoon: 11,040 hours

3. Days into monsoon season...
   ✓ days_into_sw_monsoon, days_into_ne_monsoon

✓ Temporal/seasonal features created: 10 features


In [9]:
"""
CELL 9: CREATE HISTORICAL CLIMATOLOGY FEATURES
For each date+hour, calculate historical average rainfall probability
"""

print("\n" + "="*70)
print("CREATING HISTORICAL CLIMATOLOGY FEATURES")
print("="*70)

print("\n1. Calculating historical rainfall statistics...")
print("   (For each day-of-year, what's the historical average?)")

# Group by day_of_year and hour to get historical averages
historical = df.groupby(['day_of_year', 'hour']).agg({
    'precipitation': [
        ('hist_rain_prob', lambda x: (x > 0.1).mean()),
        ('hist_rain_mean', 'mean')
    ],
    'temperature_2m': [
        ('hist_temp_mean', 'mean')
    ]
}).reset_index()

# Flatten columns
historical.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                      for col in historical.columns.values]
historical.columns = ['day_of_year', 'hour', 'historical_rain_probability', 
                      'historical_rain_mean', 'historical_temp_mean']

# Merge back to main dataframe
df = df.merge(historical, on=['day_of_year', 'hour'], how='left', suffixes=('', '_hist'))

print(f"   ✓ historical_rain_probability (0-1)")
print(f"   ✓ historical_rain_mean (mm)")
print(f"   ✓ historical_temp_mean (°C)")

# 2. ANOMALY: Current vs Historical
print("\n2. Calculating anomalies (current vs historical)...")
df['rainfall_anomaly_vs_historical'] = df['precipitation'] - df['historical_rain_mean']
df['temp_anomaly_vs_historical'] = df['temp_current'] - df['historical_temp_mean']
print(f"   ✓ rainfall_anomaly_vs_historical")
print(f"   ✓ temp_anomaly_vs_historical")

climatology_features = ['historical_rain_probability', 'historical_rain_mean', 
                       'historical_temp_mean', 'rainfall_anomaly_vs_historical',
                       'temp_anomaly_vs_historical']

print(f"\n✓ Historical climatology features created: {len(climatology_features)} features")



CREATING HISTORICAL CLIMATOLOGY FEATURES

1. Calculating historical rainfall statistics...
   (For each day-of-year, what's the historical average?)
   ✓ historical_rain_probability (0-1)
   ✓ historical_rain_mean (mm)
   ✓ historical_temp_mean (°C)

2. Calculating anomalies (current vs historical)...
   ✓ rainfall_anomaly_vs_historical
   ✓ temp_anomaly_vs_historical

✓ Historical climatology features created: 5 features


In [10]:
"""
CELL 10: CREATE DERIVED ATMOSPHERIC FEATURES
Physically meaningful features not in raw data
"""

print("\n" + "="*70)
print("CREATING DERIVED ATMOSPHERIC FEATURES")
print("="*70)

# 1. PRESSURE TENDENCY (Rate of change)
print("\n1. Pressure tendency features...")
df['pressure_tendency_3h'] = (df['pressure_current'] - df['surface_pressure'].shift(3)) / 3
df['pressure_tendency_6h'] = (df['pressure_current'] - df['surface_pressure'].shift(6)) / 6
print("   ✓ pressure_tendency_3h (hPa/hour)")
print("   ✓ pressure_tendency_6h (hPa/hour)")
print("   Note: Negative = falling pressure = weather system approaching")

# 2. WIND DIRECTION FEATURES
print("\n2. Wind direction features...")

# Check if wind is from monsoon directions
df['wind_from_southwest'] = ((df['wind_direction_current'] >= 180) & 
                             (df['wind_direction_current'] <= 270)).astype(int)
df['wind_from_northeast'] = ((df['wind_direction_current'] >= 0) & 
                             (df['wind_direction_current'] <= 90)).astype(int)
print("   ✓ wind_from_southwest (SW monsoon flow)")
print("   ✓ wind_from_northeast (NE monsoon flow)")

# Wind direction change
df['wind_direction_change_6h'] = np.abs(
    df['wind_direction_current'] - df['wind_direction_10m'].shift(6)
)
# Handle wrap-around (0° and 360° are same)
df['wind_direction_change_6h'] = df['wind_direction_change_6h'].apply(
    lambda x: min(x, 360 - x) if pd.notna(x) else x
)
print("   ✓ wind_direction_change_6h (directional shift)")

# 3. INSTABILITY INDEX
print("\n3. Atmospheric instability index...")
# Higher instability = more likely convective rainfall
df['instability_index'] = (df['temp_24h_range'] * 
                           (1 - df['dew_point_depression_current'] / 20))
df['instability_index'] = df['instability_index'].clip(lower=0)  # Ensure non-negative
print("   ✓ instability_index (convective potential)")

# 4. MOISTURE FLUX
print("\n4. Moisture flux (moisture transport)...")
df['moisture_flux'] = df['humidity_24h_mean'] * df['wind_24h_mean']
print("   ✓ moisture_flux (humidity × wind)")

derived_features = ['pressure_tendency_3h', 'pressure_tendency_6h',
                   'wind_from_southwest', 'wind_from_northeast',
                   'wind_direction_change_6h', 'instability_index',
                   'moisture_flux']

print(f"\n✓ Derived atmospheric features created: {len(derived_features)} features")



CREATING DERIVED ATMOSPHERIC FEATURES

1. Pressure tendency features...
   ✓ pressure_tendency_3h (hPa/hour)
   ✓ pressure_tendency_6h (hPa/hour)
   Note: Negative = falling pressure = weather system approaching

2. Wind direction features...
   ✓ wind_from_southwest (SW monsoon flow)
   ✓ wind_from_northeast (NE monsoon flow)
   ✓ wind_direction_change_6h (directional shift)

3. Atmospheric instability index...
   ✓ instability_index (convective potential)

4. Moisture flux (moisture transport)...
   ✓ moisture_flux (humidity × wind)

✓ Derived atmospheric features created: 7 features


In [11]:
"""
CELL 11: DATA LEAKAGE CHECK
Verify no features use future information to predict future
"""

print("\n" + "="*70)
print("DATA LEAKAGE VALIDATION CHECK")
print("="*70)

print("\nChecking for data leakage...")
print("Rule: Features at time T should only use data from T and earlier, NOT future")

# Test case: Pick a random row
test_idx = 1000
test_row = df.iloc[test_idx]

print(f"\nTest case: Row {test_idx}")
print(f"  Datetime: {test_row['datetime']}")
print(f"  Target (next 24h rain): {test_row['target_24h_rainfall']:.2f} mm")
print(f"  Current hour precipitation: {test_row['precipitation']:.2f} mm")

# Check 1: Rolling features should use PAST data
print(f"\n✓ CHECK 1: Rolling features use past data")
print(f"  precip_24h_sum: {test_row['precip_24h_sum']:.2f} mm (last 24h)")
print(f"  temp_24h_mean: {test_row['temp_24h_mean']:.2f} °C (last 24h)")
print(f"  Verify: These should NOT include future data")

# Check 2: Target should NOT be in features
print(f"\n✓ CHECK 2: Target not included in features")
feature_cols = [col for col in df.columns if col not in ['target_24h_rainfall', 'datetime', 
                                                           'date', 'date_dt', 'monsoon_season']]
assert 'target_24h_rainfall' not in feature_cols, "❌ LEAKAGE: Target in features!"
print(f"  ✓ Target is separate from features")

# Check 3: Lag features should be shifted correctly
print(f"\n✓ CHECK 3: Lag features correctly shifted")
print(f"  rainfall_lag_1d: {test_row['rainfall_lag_1d']:.2f} mm (yesterday)")
print(f"  Current date: {test_row['date']}")

# Verify by looking at yesterday's data
yesterday_date = test_row['date_dt'] - timedelta(days=1)
yesterday_rain = daily_agg[daily_agg['date_dt'] == yesterday_date]['daily_rainfall_total'].values
if len(yesterday_rain) > 0:
    print(f"  Yesterday's actual rain: {yesterday_rain[0]:.2f} mm")
    if abs(test_row['rainfall_lag_1d'] - yesterday_rain[0]) < 0.01:
        print(f"  ✓ Lag_1d matches yesterday's data correctly!")
    else:
        print(f"  ⚠ Warning: Mismatch detected")

# Check 4: Current hour precipitation not in target calculation
print(f"\n✓ CHECK 4: Current hour not in target")
print(f"  Current hour rain: {test_row['precipitation']:.2f} mm")
print(f"  Target (next 24h): {test_row['target_24h_rainfall']:.2f} mm")
print(f"  These should be independent (current hour starts future window)")

# Check 5: Sample correlation check
print(f"\n✓ CHECK 5: Correlation between features and target")
correlation_sample = df[['target_24h_rainfall', 'precip_24h_sum', 'rainfall_lag_1d', 
                         'humidity_24h_mean', 'pressure_24h_trend']].corr()['target_24h_rainfall'].sort_values(ascending=False)
print(correlation_sample)
print(f"  Note: Past rain features should correlate, but not perfectly (no leakage)")

print(f"\n{'='*70}")
print(f"DATA LEAKAGE CHECK COMPLETE")
print(f"{'='*70}")
print(f"If all checks passed, features are properly constructed!")



DATA LEAKAGE VALIDATION CHECK

Checking for data leakage...
Rule: Features at time T should only use data from T and earlier, NOT future

Test case: Row 1000
  Datetime: 2021-02-11 16:00:00
  Target (next 24h rain): 0.00 mm
  Current hour precipitation: 0.00 mm

✓ CHECK 1: Rolling features use past data
  precip_24h_sum: 0.00 mm (last 24h)
  temp_24h_mean: 25.35 °C (last 24h)
  Verify: These should NOT include future data

✓ CHECK 2: Target not included in features
  ✓ Target is separate from features

✓ CHECK 3: Lag features correctly shifted
  rainfall_lag_1d: 0.00 mm (yesterday)
  Current date: 2021-02-11
  Yesterday's actual rain: 0.00 mm
  ✓ Lag_1d matches yesterday's data correctly!

✓ CHECK 4: Current hour not in target
  Current hour rain: 0.00 mm
  Target (next 24h): 0.00 mm
  These should be independent (current hour starts future window)

✓ CHECK 5: Correlation between features and target
target_24h_rainfall    1.000000
precip_24h_sum         0.475917
rainfall_lag_1d       

In [12]:
"""
CELL 12: REMOVE WARM-UP PERIOD AND HANDLE MISSING VALUES
Remove initial rows where lag features are NaN
"""

print("\n" + "="*70)
print("HANDLING MISSING VALUES AND WARM-UP PERIOD")
print("="*70)

# Check missing values
print("\n1. Missing values before cleanup:")
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(f"   Total missing values: {missing_counts.sum():,}")
print(f"   Columns with missing: {len(missing_cols)}")

if len(missing_cols) > 0:
    print(f"\n   Top 10 columns with most missing:")
    for col, count in missing_cols.head(10).items():
        pct = count / len(df) * 100
        print(f"     {col}: {count:,} ({pct:.2f}%)")

# Strategy: Remove first 30 days (warm-up period for lag features)
print(f"\n2. Removing warm-up period...")
initial_len = len(df)
min_date = df['datetime'].min()
warm_up_date = min_date + timedelta(days=30)

print(f"   Initial date: {min_date}")
print(f"   Warm-up cutoff: {warm_up_date}")

df = df[df['datetime'] >= warm_up_date].copy()

print(f"   Records removed: {initial_len - len(df):,}")
print(f"   Records remaining: {len(df):,}")

# Check missing after removal
missing_after = df.isnull().sum().sum()
print(f"\n3. Missing values after warm-up removal: {missing_after:,}")

# Fill any remaining missing with forward fill then backward fill
if missing_after > 0:
    print(f"\n4. Filling remaining missing values...")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill')
    
    missing_final = df.isnull().sum().sum()
    print(f"   Missing after filling: {missing_final:,}")
    
    if missing_final > 0:
        print(f"Still {missing_final} missing - will drop these rows")
        df = df.dropna()

# Remove rows without valid target
df = df[df['target_24h_rainfall'].notna()].copy()

print(f"\n✓ Data cleaning complete")
print(f"  Final dataset size: {len(df):,} records")
print(f"  Date range: {df['datetime'].min()} to {df['datetime'].max()}")



HANDLING MISSING VALUES AND WARM-UP PERIOD

1. Missing values before cleanup:
   Total missing values: 2,411
   Columns with missing: 50

   Top 10 columns with most missing:
     temp_trend_14d: 336 (0.77%)
     rainfall_lag_14d: 336 (0.77%)
     rainfall_lag_7d: 168 (0.38%)
     temp_trend_7d: 168 (0.38%)
     temp_lag_7d: 168 (0.38%)
     pressure_change_7d: 168 (0.38%)
     pressure_change_3d: 72 (0.16%)
     rainfall_lag_3d: 72 (0.16%)
     humidity_lag_3d: 72 (0.16%)
     temp_lag_3d: 72 (0.16%)

2. Removing warm-up period...
   Initial date: 2021-01-01 00:00:00
   Warm-up cutoff: 2021-01-31 00:00:00
   Records removed: 720
   Records remaining: 43,104

3. Missing values after warm-up removal: 24

4. Filling remaining missing values...
   Missing after filling: 0

✓ Data cleaning complete
  Final dataset size: 43,104 records
  Date range: 2021-01-31 00:00:00 to 2025-12-31 23:00:00


In [13]:
import os

"""
CELL 13: FEATURE SUMMARY AND SAVE FINAL FEATURE SET
Count all features and save processed data
"""

print("\n" + "="*70)
print("FEATURE ENGINEERING SUMMARY")
print("="*70)

# Identify feature columns (exclude metadata and target)
exclude_cols = [
    'datetime', 'date', 'date_dt', 'year', 'monsoon_season',
    'target_24h_rainfall', 'precipitation', 'temperature_2m',
    'relative_humidity_2m', 'dew_point_2m', 'surface_pressure',
    'cloud_cover', 'wind_speed_10m', 'wind_direction_10m',

    # ── Cell 5: Full calendar-day aggregates (include future hours) ──
    'daily_rainfall_total', 'daily_rainfall_max_hourly', 'daily_rainy_hours',
    'daily_temp_mean', 'daily_temp_max', 'daily_temp_min', 'daily_temp_range',
    'daily_humidity_mean', 'daily_humidity_max', 'daily_humidity_min',
    'daily_pressure_mean', 'daily_pressure_min', 'daily_pressure_max',
    'daily_wind_mean', 'daily_wind_max',
    'daily_cloud_mean', 'daily_cloud_max',
    'daily_dewpoint_mean',

    # ── Cell 6: Derived from today's daily total (inherit leakage) ──
    'rainfall_cumsum_3d', 'rainfall_cumsum_7d',
    'rainfall_cumsum_14d', 'rainfall_cumsum_30d',
    'rainfall_rolling_mean_7d', 'rainfall_rolling_mean_14d', 'rainfall_rolling_mean_30d',
    'consecutive_dry_days', 'consecutive_rainy_days', 'days_since_last_rain',
    'temp_trend_7d', 'temp_trend_14d',
    'pressure_change_3d', 'pressure_change_7d',

    # ── Cell 9: Historical stats computed on full dataset (incl. test period) ──
    'historical_rain_probability', 'historical_rain_mean', 'historical_temp_mean',
    'rainfall_anomaly_vs_historical', 'temp_anomaly_vs_historical',
]

feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Clean feature count: {len(feature_cols)}")  # → 71

# Categorize features
feature_categories = {
    'Rolling 24h': [c for c in feature_cols if '24h' in c],
    'Current State': [c for c in feature_cols if 'current' in c or c in ['hour_sin', 'hour_cos']],
    'Daily Lags': [c for c in feature_cols if 'lag' in c and 'd' in c],
    'Cumulative': [c for c in feature_cols if 'cumsum' in c or 'rolling_mean' in c],
    'Consecutive': [c for c in feature_cols if 'consecutive' in c or 'days_since' in c],
    'Temporal': [c for c in feature_cols if any(x in c for x in ['month', 'day_of_year', 'season', 'monsoon'])],
    'Trends': [c for c in feature_cols if 'trend' in c or 'change' in c or 'tendency' in c],
    'Historical': [c for c in feature_cols if 'historical' in c or 'anomaly' in c],
    'Derived': [c for c in feature_cols if any(x in c for x in ['instability', 'moisture_flux', 'wind_from'])]
}

print(f"\nFEATURE BREAKDOWN:")
print(f"{'='*70}")
total = 0
for category, features in feature_categories.items():
    count = len(features)
    total += count
    print(f"  {category:<20}: {count:>3} features")
    
print(f"  {'='*45}")
print(f"  {'TOTAL FEATURES':<20}: {total:>3} features")
print(f"{'='*70}")

# Show sample of features
print(f"\nSAMPLE FEATURES:")
for category, features in list(feature_categories.items())[:3]:
    print(f"\n  {category}:")
    for feat in features[:5]:
        print(f"    - {feat}")
    if len(features) > 5:
        print(f"    ... and {len(features)-5} more")

# Dataset info
print(f"\nDATASET INFO:")
print(f"  Total records: {len(df):,}")
print(f"  Total features: {len(feature_cols)}")
print(f"  Date range: {df['datetime'].min().date()} to {df['datetime'].max().date()}")
print(f"  Target variable: target_24h_rainfall")

# Save processed data
output_file = 'data/hourly_features_engineered_2021_2025.csv'
df.to_csv(output_file, index=False)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)

print(f"\n{'='*70}")
print(f"FEATURE ENGINEERING COMPLETE!")
print(f"{'='*70}")
print(f"✓ File saved: {output_file}")
print(f"  Size: {file_size_mb:.2f} MB")
print(f"  Records: {len(df):,}")
print(f"  Features: {len(feature_cols)}")
print(f"\nReady for train/test split and modeling!")



FEATURE ENGINEERING SUMMARY
✅ Clean feature count: 71

FEATURE BREAKDOWN:
  Rolling 24h         :  28 features
  Current State       :  10 features
  Daily Lags          :  12 features
  Cumulative          :   0 features
  Consecutive         :   0 features
  Temporal            :  12 features
  Trends              :   5 features
  Historical          :   0 features
  Derived             :   4 features
  TOTAL FEATURES      :  71 features

SAMPLE FEATURES:

  Rolling 24h:
    - precip_24h_sum
    - precip_24h_max
    - precip_24h_mean
    - precip_24h_std
    - precip_24h_rainy_hours
    ... and 23 more

  Current State:
    - temp_current
    - humidity_current
    - pressure_current
    - dewpoint_current
    - wind_speed_current
    ... and 5 more

  Daily Lags:
    - rainfall_lag_1d
    - rainfall_lag_2d
    - rainfall_lag_3d
    - rainfall_lag_7d
    - rainfall_lag_14d
    ... and 7 more

DATASET INFO:
  Total records: 43,104
  Total features: 71
  Date range: 2021-01-31 to 2025